In [1]:
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import multivariate_normal
from datetime import timedelta

np.random.seed(42)

# ============================================================
# CONFIG
# ============================================================

N_CUSTOMERS = 150_000
WEEKS = 120
START_DATE = pd.to_datetime("2023-01-01")

# ============================================================
# 1️⃣ CUSTOMER MASTER (Correlated Structure)
# ============================================================

def generate_customer_master(n):

    # Correlated latent variables: income, age, risk_factor
    mean = [12, 40, 0]  # log-income, age, latent risk
    cov = [
        [1.2, -0.3, -0.5],
        [-0.3, 25, 0.2],
        [-0.5, 0.2, 1]
    ]

    latent = multivariate_normal(mean, cov).rvs(n)

    log_income = latent[:,0]
    age = np.clip(latent[:,1], 21, 75)
    latent_risk = latent[:,2]

    income = np.exp(log_income) * 1000
    credit_score = np.clip(750 + 40*(log_income - log_income.mean()) - 60*latent_risk, 300, 900)
    risk_score = expit(latent_risk)

    digital_affinity = np.clip(1 - (age - 21)/60 + np.random.normal(0,0.1,n), 0,1)
    tenure = np.random.randint(6, 180, n)

    credit_limit = income * np.random.uniform(0.2, 0.5, n)

    df = pd.DataFrame({
        "customer_id": [f"CUST_{i}" for i in range(n)],
        "age": age,
        "annual_income": income,
        "credit_score": credit_score,
        "risk_score": risk_score,
        "digital_affinity": digital_affinity,
        "tenure_months": tenure,
        "credit_limit": credit_limit
    })

    return df

# ============================================================
# 2️⃣ TRANSACTION ENGINE (Heavy-Tailed + Utilization)
# ============================================================

def generate_transactions(customers):

    txn_rows = []

    for _, row in customers.iterrows():

        base_spend = np.random.lognormal(mean=10, sigma=1.0)
        utilization = base_spend / row.credit_limit

        delinquency_prob = expit(4*utilization + 3*row.risk_score - 3)
        delinquency_flag = np.random.binomial(1, delinquency_prob)

        n_txn = np.random.poisson(20)

        for _ in range(n_txn):
            txn_rows.append([
                row.customer_id,
                START_DATE + timedelta(days=int(np.random.uniform(0,365))),
                np.random.lognormal(mean=8, sigma=1.2),
                utilization,
                delinquency_flag
            ])

    txn_df = pd.DataFrame(txn_rows, columns=[
        "customer_id",
        "txn_date",
        "txn_amount",
        "utilization_ratio",
        "delinquency_flag"
    ])

    return txn_df

# ============================================================
# 3️⃣ DIGITAL ENGAGEMENT
# ============================================================

def generate_digital(customers):

    df = customers.copy()

    df["app_sessions_30d"] = (
        df.digital_affinity * 35 + np.random.normal(0,5,len(df))
    ).astype(int)

    df["email_open_rate"] = np.clip(
        df.digital_affinity + np.random.normal(0,0.1,len(df)),0,1
    )

    df["complaints"] = np.random.poisson(df.risk_score * 3)

    return df[[
        "customer_id",
        "app_sessions_30d",
        "email_open_rate",
        "complaints"
    ]]

# ============================================================
# 4️⃣ CHURN SIMULATION
# ============================================================

def generate_churn(customers, digital):

    merged = customers.merge(digital, on="customer_id")

    logit = (
        -2.5
        + 3*merged.risk_score
        + 2*(merged.complaints > 2)
        - 0.02*merged.tenure_months
        - 1.5*merged.digital_affinity
    )

    churn_prob = expit(logit)
    merged["churn_flag"] = np.random.binomial(1, churn_prob)

    return merged[["customer_id", "churn_flag"]]

# ============================================================
# 5️⃣ MMM ENGINE (Correlated + Adstock + Saturation + Shock)
# ============================================================

def adstock(x, decay=0.6):
    result = np.zeros_like(x)
    for t in range(len(x)):
        result[t] = x[t] + (decay * result[t-1] if t > 0 else 0)
    return result

def hill(x, alpha=1.5):
    return x**alpha / (x**alpha + 1)

def generate_mmm(weeks):

    mean = [10, 8, 7]
    cov = [
        [1, 0.6, 0.4],
        [0.6, 1, 0.5],
        [0.4, 0.5, 1]
    ]

    latent_spend = multivariate_normal(mean, cov).rvs(weeks)
    tv = np.exp(latent_spend[:,0]) * 1e5
    meta = np.exp(latent_spend[:,1]) * 8e4
    search = np.exp(latent_spend[:,2]) * 5e4

    seasonality = 1 + 0.2*np.sin(np.linspace(0, 4*np.pi, weeks))

    macro = np.ones(weeks)
    macro[50:60] *= 0.85   # recession shock

    competitor = np.ones(weeks)
    competitor[70:80] *= 1.2

    tv_ad = adstock(tv)
    meta_ad = hill(adstock(meta))
    search_ad = search

    sales = (
        0.00002 * tv_ad
        + 0.00004 * meta_ad
        + 0.00005 * search_ad
        + 2000 * seasonality
        + 1500 * macro
        - 1000 * (competitor - 1)
        + np.random.normal(0, 800, weeks)
    )

    df = pd.DataFrame({
        "week": np.arange(weeks),
        "tv_spend": tv,
        "meta_spend": meta,
        "search_spend": search,
        "seasonality": seasonality,
        "macro_index": macro,
        "competitor_index": competitor,
        "sales": sales
    })

    return df

# ============================================================
# RUN FULL PIPELINE
# ============================================================

customers = generate_customer_master(N_CUSTOMERS)
transactions = generate_transactions(customers)
digital = generate_digital(customers)
churn = generate_churn(customers, digital)
mmm_weekly = generate_mmm(WEEKS)

print("Customer Master:", customers.shape)
print("Transactions:", transactions.shape)
print("Digital:", digital.shape)
print("Churn:", churn.shape)
print("MMM Weekly:", mmm_weekly.shape)

Customer Master: (150000, 8)
Transactions: (3001170, 5)
Digital: (150000, 4)
Churn: (150000, 2)
MMM Weekly: (120, 8)
